In [ ]:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME/blob/main/kmeans_clustering.ipynb)

In [ ]:
import os
import sys

# Detect environment and set project root directory
if 'google.colab' in sys.modules:
    print("Running in Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')
    # If running in Colab, point to your mounted drive directory
    ROOT_DIR = "/content/drive/My Drive/Colab Notebooks/Pacific Dataviz Challenge 2026/"
else:
    print("Running in Local / GitHub environment")
    # Dynamically sets root to wherever this notebook file lives
    ROOT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()

Current environment: Google Colab Virtual Machine
Mounted at /content/drive


In [ ]:
# Data architecture

# Raw data
RAW_DATA_DIR = os.path.join(ROOT_DIR, "data", "raw")

# Processed data
PROCESSED_DATA_DIR = os.path.join(ROOT_DIR, "data", "processed")

# Models
MODELS_DIR = os.path.join(ROOT_DIR, "models")

# Verification print
print(f"Target Root Directory successfully mapped to: {ROOT_DIR}")
print(f"Raw Data Directory mapped to: {RAW_DATA_DIR}")
print(f"Processed Data Directory mapped to: {PROCESSED_DATA_DIR}")
print(f"Models Directory mapped to: {MODELS_DIR}")

Target Root Directory successfully mapped to: /content/drive/My Drive/Colab Notebooks/Pacific Dataviz Challenge 2026/
Raw Data Directory mapped to: /content/drive/My Drive/Colab Notebooks/Pacific Dataviz Challenge 2026/data/raw
Processed Data Directory mapped to: /content/drive/My Drive/Colab Notebooks/Pacific Dataviz Challenge 2026/data/processed
Models Directory mapped to: /content/drive/My Drive/Colab Notebooks/Pacific Dataviz Challenge 2026/models


In [ ]:
# Import pandas for data processing
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
# Import data sets downloaded from Pacific Data Hub’s .Stat Explorer

# Import tourism arrivals data - did not use this as more data was available in disaggregated path
tourism_path = os.path.join(RAW_DATA_DIR, "pacific_data_tourist_arrivals.csv")
tourism_df = pd.read_csv(tourism_path)

# Import tourism disagrregated path
disaggregated_path = os.path.join(RAW_DATA_DIR, "pacific_data_tourist_arrivals_disaggregated.csv")
disaggregated_df = pd.read_csv(disaggregated_path)

# Import population percentage data
population_path = os.path.join(RAW_DATA_DIR, "pacific_data_population_change.csv")
population_df = pd.read_csv(population_path)

# Import renewable energy percentage data
renewable_path = os.path.join(RAW_DATA_DIR, "pacific_data_renewable_share.csv")
renewable_df = pd.read_csv(renewable_path)

# Import total energy generation
energy_path = os.path.join(RAW_DATA_DIR, "pacific_data_energy_generation.csv")
energy_df = pd.read_csv(energy_path)

In [ ]:
# Drop excess columns
def df_drop_cols (df):
  columns_to_keep = ['Pacific Island Countries and territories', 'TIME_PERIOD', 'OBS_VALUE']
  df = df[columns_to_keep]
  df.rename(columns={'Pacific Island Countries and territories': 'Location', 'TIME_PERIOD': 'Year', 'OBS_VALUE': 'Value'}, inplace=True)
  return df

In [ ]:
tourism_df = df_drop_cols(tourism_df)
population_df = df_drop_cols(population_df)
renewable_df = df_drop_cols(renewable_df)
energy_df = df_drop_cols(energy_df)

/tmp/ipykernel_1491/3576205078.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'Pacific Island Countries and territories': 'Location', 'TIME_PERIOD': 'Year', 'OBS_VALUE': 'Value'}, inplace=True)
/tmp/ipykernel_1491/3576205078.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'Pacific Island Countries and territories': 'Location', 'TIME_PERIOD': 'Year', 'OBS_VALUE': 'Value'}, inplace=True)
/tmp/ipykernel_1491/3576205078.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.o

In [ ]:
disaggregated_df.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ',
       'Frequency', 'GEO_PICT', 'Pacific Island Countries and territories',
       'VISITOR_DURATION_CAT', 'Visitor duration category', 'TIME_PERIOD',
       'Time', 'OBS_VALUE', 'Observation value', 'OBS_STATUS',
       'Observation Status', 'UNIT_MEASURE', 'Unit of measure', 'ERROR_TYPE',
       'Error type', 'ERROR_VAL', 'Error value', 'REPORTING_TYPE',
       'Reporting Type', 'OBS_COMMENT', 'Comment'],
      dtype='object')

In [ ]:
disaggregated_cols_to_keep = ['Pacific Island Countries and territories', 'VISITOR_DURATION_CAT', 'TIME_PERIOD', 'OBS_VALUE']
disaggregated_df = disaggregated_df[disaggregated_cols_to_keep]
disaggregated_df.rename(columns={'Pacific Island Countries and territories': 'Location', 'VISITOR_DURATION_CAT': 'Tourist', 'TIME_PERIOD': 'Year', 'OBS_VALUE': 'Value'}, inplace=True)

In [ ]:
disaggregated_df.shape

(707, 4)

In [ ]:
aggregted_df = disaggregated_df.groupby(['Location', 'Year'], as_index=False)['Value'].sum()
aggregted_df

,Location,Year,Value
0,American Samoa,1995,34000
1,American Samoa,1996,35000
2,American Samoa,1997,26000
3,American Samoa,1998,36000
4,American Samoa,1999,41000
...,...,...,...
485,Vanuatu,2017,332800
486,Vanuatu,2018,350200
487,Vanuatu,2019,256000
488,Vanuatu,2020,82400


In [ ]:
tourism_aggregated_df = aggregted_df.copy()

In [ ]:
# Determine if there are differences between the locations present in each dataset.
dfs = [population_df, renewable_df, energy_df, tourism_aggregated_df]
df_ref = ['population_df', 'renewable_df', 'energy_df', 'tourism_aggregated_df']
locations = []
i = 0

print('Count unique locations in each df:')
print()

for df in dfs:
  name = df_ref[i]
  # Store as a set for comparison
  locations.append(set(df['Location'].unique()))
  unique_locations = (df['Location'].unique())
  print(f'{name} contains {len(unique_locations)} unique locations.')
  i += 1

print()
print('-'*10)
print()

# Check if all sets are identical
if all(loc_set == locations[0] for loc_set in locations):
    print("The dfs all have the same locations in.")
else:
    print("The dfs have different locations and need filtering.")


Count unique locations in each df:

population_df contains 22 unique locations.
renewable_df contains 20 unique locations.
energy_df contains 18 unique locations.
tourism_aggregated_df contains 18 unique locations.

----------

The dfs have different locations and need filtering.


In [ ]:
# Find the intersection of all location sets to get common locations
common_locations = list(locations[0].intersection(*locations[1:]))
common_locations.sort()
print(f'The {len(common_locations)} locations present across all dfs are: {common_locations}')

The 15 locations present across all dfs are: ['Cook Islands', 'Fiji', 'French Polynesia', 'Guam', 'Kiribati', 'Marshall Islands', 'New Caledonia', 'Niue', 'Palau', 'Papua New Guinea', 'Samoa', 'Solomon Islands', 'Tonga', 'Tuvalu', 'Vanuatu']


In [ ]:
# Filter all dfs to only include common locations
i = 0

for df in dfs:
  df = df[df['Location'].isin(common_locations)]
  dfs[i] = df
  filtered_locations = (df['Location'].unique())
  filtered_locations.sort()
  print(f'{df_ref[i]} contains {len(filtered_locations)} unique locations - {filtered_locations}.')
  print()
  print('-'*10)
  print()
  i+=1

population_df contains 15 unique locations - ['Cook Islands' 'Fiji' 'French Polynesia' 'Guam' 'Kiribati'
 'Marshall Islands' 'New Caledonia' 'Niue' 'Palau' 'Papua New Guinea'
 'Samoa' 'Solomon Islands' 'Tonga' 'Tuvalu' 'Vanuatu'].

----------

renewable_df contains 15 unique locations - ['Cook Islands' 'Fiji' 'French Polynesia' 'Guam' 'Kiribati'
 'Marshall Islands' 'New Caledonia' 'Niue' 'Palau' 'Papua New Guinea'
 'Samoa' 'Solomon Islands' 'Tonga' 'Tuvalu' 'Vanuatu'].

----------

energy_df contains 15 unique locations - ['Cook Islands' 'Fiji' 'French Polynesia' 'Guam' 'Kiribati'
 'Marshall Islands' 'New Caledonia' 'Niue' 'Palau' 'Papua New Guinea'
 'Samoa' 'Solomon Islands' 'Tonga' 'Tuvalu' 'Vanuatu'].

----------

tourism_aggregated_df contains 15 unique locations - ['Cook Islands' 'Fiji' 'French Polynesia' 'Guam' 'Kiribati'
 'Marshall Islands' 'New Caledonia' 'Niue' 'Palau' 'Papua New Guinea'
 'Samoa' 'Solomon Islands' 'Tonga' 'Tuvalu' 'Vanuatu'].

----------



In [ ]:
for df in dfs:
  print(df.describe())

              Year       Value
count   540.000000  540.000000
mean   2007.500000    0.857019
std      10.397927    1.514540
min    1990.000000   -3.870000
25%    1998.750000    0.127500
50%    2007.500000    0.850000
75%    2016.250000    1.975000
max    2025.000000    3.750000
              Year       Value
count   346.000000  346.000000
mean   2011.034682   20.883410
std       6.664547   20.862109
min    2000.000000    0.000000
25%    2005.000000    2.152500
50%    2011.000000    9.015000
75%    2017.000000   42.237500
max    2023.000000   66.380000
              Year       Value
count   360.000000   360.00000
mean   2011.500000   618.84000
std       6.931821   953.29117
min    2000.000000     2.90000
25%    2005.750000    34.60000
50%    2011.500000    83.20000
75%    2017.250000   857.10000
max    2023.000000  3508.30000
              Year         Value
count   416.000000  4.160000e+02
mean   2008.572115  2.058585e+05
std       8.140903  3.324978e+05
min    1995.000000  3.500000e+0

In [ ]:
def plot_years(dfs, df_ref):
  i = 0
  for df in dfs:
    print(df_ref[i])

    scatter = px.scatter(dfs[i], x='Location', y='Year')
    # Add a horizontal line at y=2023
    scatter.add_shape(
        type="line",
        xref="paper",
        x0=0,
        y0=2023,
        x1=1,
        y1=2023,
        line=dict(color="Red", width=2, dash="dash")
    )
    # Add a horizontal line at y=2000
    scatter.add_shape(
        type="line",
        xref="paper",
        x0=0,
        y0=2000,
        x1=1,
        y1=2000,
        line=dict(color="Red", width=2, dash="dash")
    )
    scatter.show()
    i +=1
    print()

In [ ]:
# Review presence of year per location for each df
plot_years(dfs,df_ref)

population_df



renewable_df



energy_df



tourism_aggregated_df


In [ ]:
locations_to_drop = ['Solomon Islands', 'Marshall Islands']
target_years = list(range(2000,2021))

def clean_and_trim_dataframe(df):
    df_clean = df[~df['Location'].isin(locations_to_drop)].copy()
    # Ensure 'Year' column is integer type for accurate filtering
    df_clean['Year'] = df_clean['Year'].astype(int)
    # Use explicit range filtering for years
    df_clean = df_clean[(df_clean['Year'] >= min(target_years)) & (df_clean['Year'] <= max(target_years))]
    return df_clean

In [ ]:
# Clean the dfs,
clean_dfs = dfs.copy()
for i, df in enumerate(clean_dfs):
  clean_dfs[i] = clean_and_trim_dataframe(df)
  clean_dfs[i].reset_index(drop=True, inplace=True)
  clean_dfs[i].sort_values(by=['Location'], inplace=True)
  print(clean_dfs[i].describe())

              Year       Value
count   273.000000  273.000000
mean   2010.000000    0.759194
std       6.066422    1.350620
min    2000.000000   -3.870000
25%    2005.000000    0.040000
50%    2010.000000    0.620000
75%    2015.000000    1.680000
max    2020.000000    3.510000
              Year       Value
count   273.000000  273.000000
mean   2010.000000   19.530586
std       6.066422   21.306402
min    2000.000000    0.000000
25%    2005.000000    1.170000
50%    2010.000000    7.370000
75%    2015.000000   40.920000
max    2020.000000   66.380000
              Year        Value
count   273.000000   273.000000
mean   2010.000000   688.937729
std       6.066422   981.998125
min    2000.000000     2.900000
25%    2005.000000    30.900000
50%    2010.000000    93.400000
75%    2015.000000  1019.300000
max    2020.000000  3508.300000
              Year         Value
count   273.000000  2.730000e+02
mean   2010.000000  2.456989e+05
std       6.066422  3.512504e+05
min    2000.000000  6.

In [ ]:
# Population data source (this is a single source with estimates for all locations)
pop_source = "SPC SDD Mid-Year Population Estimates Data Sheet"
pop_url = "https://pacificdata.org/story/pacific-island-populations-2020"

# Clean 2020 Demographic Anchors for Back-Casting Loop
anchor_population_2020 = {
    'Cook Islands': 15281,
    'Fiji': 894961,
    'French Polynesia': 278908,
    'Guam': 176664,
    'Kiribati': 118744,
    'New Caledonia': 273015,
    'Niue': 1562,
    'Palau': 17930,
    'Papua New Guinea': 8934475,
    'Samoa': 19866,
    'Tonga': 99780,
    'Tuvalu': 1050,
    'Vanuatu': 294688
}

In [ ]:
df_pop = clean_dfs[0].copy()

# Convert Value column to decimals
df_pop['Growth_Decimal'] = df_pop['Value'] / 100.0

# Create a list to store reconstructed country slices
reconstructed_slices = []

# Loop through each country in anchor list
for country, anchor_pop in anchor_population_2020.items():

    # Extract rows for specific country
    df_c = df_pop[df_pop['Location'] == country].copy()
    if df_c.empty:
        continue

    # Initialize calculation dictionary
    calculated_pops = {2020: anchor_pop}

    # Sort past years in descending order
    past_years = sorted([year for year in df_c['Year'].unique() if year < 2020], reverse=True)

    # Set starting point for backward chain
    current_pop = anchor_pop

    for year in past_years:
        try:
            # Look up the growth rate that occurred to get to the next year
            growth_rate = df_c.loc[df_c['Year'] == year + 1, 'Growth_Decimal'].values[0]

            # Divide the next year's population by growth factor
            current_pop = current_pop / (1 + growth_rate)
            calculated_pops[year] = current_pop

        except IndexError:
            # Fallback if a specific year row is missing in the data matrix
            calculated_pops[year] = np.nan

    # Map calculated timeline dictionary back to this country's rows
    df_c['Calculated_Population'] = df_c['Year'].map(calculated_pops)
    reconstructed_slices.append(df_c)

# Combine all processed country slices back into single df
clean_dfs[0] = pd.concat(reconstructed_slices).sort_values(by=['Location', 'Year']).reset_index(drop=True)

# Clean up the population column by rounding to whole numbers
clean_dfs[0]['Calculated_Population'] = clean_dfs[0]['Calculated_Population'].round(0)

# Drop temporary decimal helper column
clean_dfs[0] = clean_dfs[0].drop(columns=['Value', 'Growth_Decimal'])
clean_dfs[0]['Calculated_Population'] = clean_dfs[0]['Calculated_Population'].astype(int)

In [ ]:
# Rename value columns across other three dfs
clean_dfs[1].rename(columns={'Value': 'Renewable_Share'}, inplace=True)
clean_dfs[2].rename(columns={'Value': 'Energy_Generation'}, inplace=True)
clean_dfs[3].rename(columns={'Value': 'Tourism_Aggregated_Arrivals'}, inplace=True)

In [ ]:
# Merge clean_dfs into a master df
master_df = pd.merge(clean_dfs[0], clean_dfs[1], on=['Location', 'Year'], how='outer')
master_df = pd.merge(master_df, clean_dfs[2], on=['Location', 'Year'], how='outer')
master_df = pd.merge(master_df, clean_dfs[3], on=['Location', 'Year'], how='outer')

In [ ]:
from sklearn.preprocessing import StandardScaler

# Resolve value columns which are heavily right skewed
# Log transform raw features (mitigate right skews)
master_df['Log_Population'] = np.log10(master_df['Calculated_Population'])
# Log_Energy and Log_Tourism are not needed for K-Means, but are needed for Piecewise
master_df['Log_Energy'] = np.log10(master_df['Energy_Generation'])
master_df['Log_Tourism'] = np.log10(master_df['Tourism_Aggregated_Arrivals'])

# Division of features by population means per island scale cancels out magnitudal differences
# Feature engineering of per-capita ratios from raw features
master_df['Tourism_Per_Capita'] = master_df['Tourism_Aggregated_Arrivals'] / master_df['Calculated_Population']
master_df['Energy_Per_Capita'] = master_df['Energy_Generation'] / master_df['Calculated_Population']

# These are to be used in piecwise linear regression
# They are not needed in K-Means, as the Log_Population, Tourism_Per_Capita, Energy_Per_Capita values are already present
master_df['Total_Human_Load'] = master_df['Calculated_Population'] + master_df['Tourism_Aggregated_Arrivals']
master_df['Log_Human_Load'] = np.log10(master_df['Total_Human_Load'])
master_df['Log_Energy_Per_Capita'] = np.log10(master_df['Energy_Per_Capita'])

In [ ]:
# Export preprocessed data
output_csv_path = os.path.join(PROCESSED_DATA_DIR, "master_df.csv")

# Save to csv
master_df.to_csv(output_csv_path, index=False)